# SPAGHETTI-1 — Data pipeline (local machine)

This notebook prepares the training data: it removes duplicated and contradictory images, flags the noisy ones and splits the clean set into cross-validation folds.

Every threshold and rule used here was chosen in the dataset-study notebook `AN2DL_Challenge_26_27.ipynb`. Only the computations needed for cleaning are repeated, without plots, so this notebook runs on its own.

The twin notebook `AN2DL_Pipeline_Colab.ipynb` contains the same code for Google Colab; only the setup cell differs.

**Requirements (once):**
1. A Python 3.11 or 3.12 environment (PyTorch does not support very recent versions such as 3.15 yet), e.g. `py -3.12 -m venv .venv`.
2. `pip install kaggle numpy pandas pillow scikit-learn ipykernel`.
3. The Kaggle token saved in `~/.kaggle/access_token` (on Windows `C:\Users\<user>\.kaggle\access_token`).
4. Select that environment as the notebook kernel in VS Code.

In [2]:
# ---------------------------------------------------------------
# 0. Colab setup: Kaggle token from the Colab Secrets (key KAGGLE_TOKEN),
# download and extraction of the competition data into /content.
# The download is skipped if the runtime already has the data.
# ---------------------------------------------------------------
!pip install --upgrade kaggle

import glob
import os
import zipfile
from pathlib import Path

from google.colab import userdata

DATA_DIR = Path("/content")   # train/, test/ and the CSVs end up here
N_WORKERS = 8                 # threads used to read the images in parallel

if not (DATA_DIR / "train.csv").exists():
    token_path = os.path.expanduser("~/.kaggle/access_token")
    os.makedirs(os.path.dirname(token_path), exist_ok=True)
    with open(token_path, "w") as f:
        f.write(userdata.get("KAGGLE_TOKEN"))
    os.chmod(token_path, 0o600)   # the token must be readable only by its owner
    !kaggle competitions download -c an2dl2627 -p /content/
    with zipfile.ZipFile(glob.glob("/content/*.zip")[0]) as archive:
        archive.extractall("/content/")

print(f"Data folder: {DATA_DIR}")

  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2
100% 2.51G/2.51G [00:26<00:00, 103MB/s] 

Data folder: /content


## 1. Configuration
Imports, dataset constants and the thresholds chosen in the dataset study (EDA notebook, sections 10–13).

In [3]:
# ---------------------------------------------------------------
# Imports and configuration shared by all the cells below.
# DATA_DIR and N_WORKERS are defined in the setup cell above.
# ---------------------------------------------------------------
import json
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42                     # fixed seed -> reproducible folds
IMAGE_SIZE = (256, 256)       # (width, height) expected for every PNG

# CSV listing the ids of each split (test ids come from the sample submission)
SPLIT_CSVS = {"train": "train.csv", "test": "sample_submission.csv"}
EXPECTED_COUNTS = {"train": 4328, "test": 14736}

CLASS_NAMES = {
    0: "Spaghetti alla rinfusa",   # irregular / disturbed
    1: "Gemelli",                  # merging pair
    2: "Gnocchi",                  # round smooth
    3: "Ravioli",                  # in-between smooth
    4: "Penne",                    # cigar-shaped smooth
    5: "Fusilli con la sbarra",    # barred spiral
    6: "Fusilli stretti",          # tight spiral
    7: "Riccioli",                 # loose spiral
    8: "Linguine",                 # edge-on disk
    9: "Lasagna",                  # edge-on disk with bulge
}

# Thresholds chosen in the dataset study (EDA notebook)
HP_SIDE = 32             # side of the high-pass descriptor (near-duplicate search)
CHUNK = 256              # train images compared per block (limits memory usage)
GROUP_THRESHOLD = 0.97   # similarity >= 0.97 -> same group (kept in the same fold)
DEDUP_THRESHOLD = 0.99   # similarity >= 0.99 -> confident copy (keep only one)
COLOR_Z = 5.0            # colorfulness robust z-score above which an image is 'noisy'
N_FOLDS = 5              # cross-validation folds

## 2. Manifest and file layout validation
Same checks as in the EDA notebook: correct CSV headers, well-formed unique ids, labels 0–9, official counts and one PNG for every id.

In [4]:
# ---------------------------------------------------------------
# Read both CSVs into a single table (the "manifest") and check that
# CSVs and image folders are consistent.
# ---------------------------------------------------------------
def load_manifest(data_dir: Path) -> pd.DataFrame:
    """Return one table with columns id, label, split. Test labels are unknown, so they are set to NA."""
    splits = []
    for split, csv_name in SPLIT_CSVS.items():
        frame = pd.read_csv(data_dir / csv_name, dtype={"id": "string"})
        if list(frame.columns) != ["id", "label"]:
            raise ValueError(f"{csv_name}: expected the header id,label")
        # every id must look like train_00000 / test_00000
        if not frame["id"].str.fullmatch(rf"{split}_[0-9]{{5}}").fillna(False).all():
            raise ValueError(f"{csv_name}: missing or malformed image IDs")
        if frame.empty or frame["id"].duplicated().any():
            raise ValueError(f"{csv_name}: empty manifest or duplicate image IDs")
        if split == "train" and not frame["label"].isin(list(CLASS_NAMES)).all():
            raise ValueError(f"{csv_name}: expected integer class labels 0-9")
        if split == "test":
            frame["label"] = pd.NA
        frame["label"] = frame["label"].astype("Int64")     # integer type that also allows NA
        splits.append(frame.assign(split=split))
    return pd.concat(splits, ignore_index=True)


def validate_layout(data_dir: Path, manifest: pd.DataFrame) -> dict:
    """Check the expected number of rows and that every CSV id has exactly one PNG (and vice versa)."""
    summary = {}
    for split, expected_count in EXPECTED_COUNTS.items():
        folder = data_dir / split
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing image directory: {folder}")
        ids = set(manifest.loc[manifest["split"] == split, "id"])
        file_ids = {path.stem for path in folder.glob("*.png") if path.is_file()}
        if len(ids) != expected_count:
            raise ValueError(f"{split}: expected {expected_count:,} CSV rows, found {len(ids):,}")
        missing, unexpected = sorted(ids - file_ids), sorted(file_ids - ids)
        if missing or unexpected:
            raise ValueError(f"{split}: {len(missing):,} missing PNGs {missing[:5]}, "
                             f"{len(unexpected):,} unexpected PNGs {unexpected[:5]}")
        summary[split] = {"rows": len(ids), "png_files": len(file_ids)}
    return summary


manifest = load_manifest(DATA_DIR)
print(json.dumps(validate_layout(DATA_DIR, manifest), indent=2))

{
  "train": {
    "rows": 4328,
    "png_files": 4328
  },
  "test": {
    "rows": 14736,
    "png_files": 14736
  }
}


## 3. Image scan
Every image is read once and only what the cleaning needs is kept:
- whether the file is a valid 256×256 RGB PNG;
- its **colorfulness**, turned into a robust z-score over train + test (as in EDA section 11) → `noisy` flag;
- its **32×32 high-pass descriptor** (EDA section 10) → near-duplicate search.

In [5]:
# ---------------------------------------------------------------
# Single pass over all 19,064 images: validity, colorfulness and
# high-pass descriptor (same formulas as in the EDA notebook).
# ---------------------------------------------------------------
def read_rgb_png(source) -> np.ndarray:
    """Decode a competition PNG, check it is a 256x256 RGB PNG and return a uint8 array (256, 256, 3)."""
    with Image.open(source) as image:
        if (image.format, image.mode, image.size) != ("PNG", "RGB", IMAGE_SIZE):
            raise ValueError(f"Expected 256x256 RGB PNG, found {image.format} {image.mode} {image.size}")
        return np.asarray(image)


_BOX = np.ones(5, np.float32) / 5   # 1-D averaging kernel (box blur of width 5)

def _blur_1d(v: np.ndarray) -> np.ndarray:
    return np.convolve(np.pad(v, 2, mode="edge"), _BOX, mode="valid")

def highpass_descriptor(gray: np.ndarray) -> np.ndarray:
    """Fine-detail fingerprint of an image: downscale to HP_SIDE x HP_SIDE, subtract a blurred
    copy (keeps stars/noise, removes the smooth galaxy shape), zero-mean and unit-norm."""
    f = gray.shape[0] // HP_SIDE
    small = gray.reshape(HP_SIDE, f, HP_SIDE, f).mean(axis=(1, 3))          # block average
    blurred = np.apply_along_axis(_blur_1d, 1, np.apply_along_axis(_blur_1d, 0, small))
    hp = (small - blurred).ravel()
    hp -= hp.mean()
    norm = np.linalg.norm(hp)
    return (hp / norm if norm > 0 else hp).astype(np.float32)


def describe_image(path: Path):
    """Return (error message or None, colorfulness, high-pass descriptor) for one image."""
    hp = np.zeros(HP_SIDE * HP_SIDE, np.float32)
    try:
        pixels = read_rgb_png(path).astype(np.float32)
    except (OSError, ValueError, Image.DecompressionBombError) as exc:
        return str(exc), np.nan, hp
    r, g, b = pixels[..., 0], pixels[..., 1], pixels[..., 2]
    # colorfulness (Hasler & Suesstrunk): spread and mean of the opponent colour channels
    rg, yb = r - g, 0.5 * (r + g) - b
    colorfulness = np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
    return None, float(colorfulness), highpass_descriptor(pixels.mean(axis=2))


def robust_z(values: pd.Series) -> pd.Series:
    """(x - median) / (1.4826 * MAD): a z-score that is not distorted by the outliers."""
    median = values.median()
    mad = (values - median).abs().median()
    return (values - median) / (1.4826 * mad + 1e-9)


paths = [DATA_DIR / row.split / f"{row.id}.png" for row in manifest.itertuples()]
print(f"Scanning {len(paths):,} images with {N_WORKERS} workers...")
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    outputs = list(pool.map(describe_image, paths))

# Row i of stats and hp_desc always refers to the same image (row i of manifest)
stats = manifest.copy()
stats["error"] = [o[0] for o in outputs]
stats["colorfulness"] = [o[1] for o in outputs]
hp_desc = np.stack([o[2] for o in outputs])    # (N, 1024) float32, about 80 MB
del outputs

valid = stats[stats["error"].isna()].copy()
valid["colorfulness_z"] = robust_z(valid["colorfulness"])   # over train + test, as in the EDA
train = valid[valid["split"] == "train"]
print(f"Invalid images: {int(stats['error'].notna().sum())} | valid training images: {len(train)}")

Scanning 19,064 images with 8 workers...
Invalid images: 0 | valid training images: 4328


## 4. Near-duplicate pairs (train–train)
Every training image is compared with every other training image through the correlation of their high-pass descriptors, keeping the best of the 8 rotations/flips. Only pairs with similarity ≥ `GROUP_THRESHOLD` (0.97) are needed for cleaning. Train–test pairs are not needed here: they were used in the EDA only to measure the domain shift.

In [6]:
# ---------------------------------------------------------------
# Train-train near-duplicate search. Descriptors are zero-mean and
# unit-norm, so the dot product equals the correlation of the fine details.
# Expected (EDA): 342 pairs >= 0.97, 220 pairs >= 0.99.
# ---------------------------------------------------------------
def dihedral_variants(desc: np.ndarray) -> np.ndarray:
    """Return the 8 rotations/flips of each descriptor, shape (8, n, D)."""
    squares = desc.reshape(-1, HP_SIDE, HP_SIDE)
    variants = []
    for flipped in (squares, squares[:, :, ::-1]):
        for k in range(4):
            variants.append(np.rot90(flipped, k, axes=(1, 2)).reshape(len(squares), -1))
    return np.ascontiguousarray(np.stack(variants))

pos = train.index.to_numpy()                  # rows of hp_desc / stats of the training images
ref_t = np.ascontiguousarray(hp_desc[pos].T)  # (D, n): transposed once for fast matrix products
variants = dihedral_variants(hp_desc[pos])    # (8, n, D)

found_a, found_b, found_s = [], [], []
for start in range(0, len(pos), CHUNK):
    stop = min(start + CHUNK, len(pos))
    sim = variants[0, start:stop] @ ref_t
    for v in variants[1:]:                    # keep the best orientation for every pair
        np.maximum(sim, v[start:stop] @ ref_t, out=sim)
    rows, cols = np.nonzero(sim >= GROUP_THRESHOLD)
    keep = start + rows < cols                # each pair only once (A < B), never an image with itself
    rows, cols = rows[keep], cols[keep]
    found_a.append(pos[start + rows]); found_b.append(pos[cols]); found_s.append(sim[rows, cols])

tt_pairs = pd.DataFrame({"a": np.concatenate(found_a), "b": np.concatenate(found_b),
                         "similarity": np.concatenate(found_s)})
tt_pairs["conflicting_labels"] = (stats.loc[tt_pairs["a"], "label"].astype(int).to_numpy()
                                  != stats.loc[tt_pairs["b"], "label"].astype(int).to_numpy())
print(f"Train-train pairs >= {GROUP_THRESHOLD}: {len(tt_pairs)} "
      f"(>= {DEDUP_THRESHOLD}: {int((tt_pairs['similarity'] >= DEDUP_THRESHOLD).sum())}, "
      f"conflicting labels: {int(tt_pairs['conflicting_labels'].sum())})")

Train-train pairs >= 0.97: 342 (>= 0.99: 220, conflicting labels: 28)


## 5. Cleaning
Rules derived from the dataset study:

- **Thresholds.** In the train–train pairs the share of label conflicts is ~90% below 0.95 (the level expected for two *different* galaxies, 1 − Σ pᵢ² ≈ 88%), ~20% in [0.97, 0.99) and ~2% above 0.99. Pairs ≥ 0.99 are true copies; [0.97, 0.99) is mixed.
- `group_id` (≥ 0.97): copies are grouped so they never end up split between training and validation folds. Groups whose copies have **different labels** are dropped entirely: 21 of the 24 conflicts in [0.97, 0.99) are between "sister" classes (Gnocchi ↔ Ravioli, the two Fusilli), a sign of label noise, and the true label cannot be recovered.
- `dup_id` (≥ 0.99): confident copies with the same label → keep one per group.
- **`noisy` flag.** ~8% of the training images contain strong chromatic noise, present only in 4 classes and almost absent from the test set. They are kept but flagged; the noise will be neutralised with augmentation.

Expected result: 4,328 → 4,091 images (237 removed), 4,027 groups.

In [7]:
# ---------------------------------------------------------------
# 5.1 Groups of copies via connected components ("union-find"):
# if A~B and B~C, then A, B and C are the same photo -> same group.
#   group_id (>= 0.97): used by StratifiedGroupKFold
#   dup_id   (>= 0.99): confident copies, used by the cleaning rules (5.2)
# ---------------------------------------------------------------
def connected_groups(positions: list, pairs: pd.DataFrame, threshold: float) -> pd.Series:
    """Return, for every position, the representative (smallest position) of its group."""
    parent = {p: p for p in positions}          # at the start every image is its own group

    def find(x):                                # follow the parents up to the group representative
        while parent[x] != x:
            parent[x] = parent[parent[x]]       # path compression: makes later searches faster
            x = parent[x]
        return x

    for a, b in pairs.loc[pairs["similarity"] >= threshold, ["a", "b"]].itertuples(index=False):
        root_a, root_b = find(int(a)), find(int(b))
        if root_a != root_b:                    # merge the two groups
            parent[max(root_a, root_b)] = min(root_a, root_b)
    return pd.Series([find(p) for p in positions], index=positions)

positions = [int(p) for p in train.index]

clean = train[["id", "label"]].copy()           # one row per valid training image
clean["group_id"] = pd.factorize(connected_groups(positions, tt_pairs, GROUP_THRESHOLD))[0]
clean["dup_id"] = pd.factorize(connected_groups(positions, tt_pairs, DEDUP_THRESHOLD))[0]

for name in ("group_id", "dup_id"):
    sizes = clean.groupby(name).size()
    print(f"{name}: {int((sizes > 1).sum())} groups with copies, "
          f"{int(sizes[sizes > 1].sum())} images involved, largest group = {int(sizes.max())} images")

group_id: 210 groups with copies, 486 images involved, largest group = 6 images
dup_id: 144 groups with copies, 330 images involved, largest group = 6 images


In [8]:
# ---------------------------------------------------------------
# 5.2 Cleaning rules on the copies:
#   - copies with DIFFERENT labels (group_id, >= 0.97) -> drop the whole group
#   - copies with the SAME label (dup_id, >= 0.99)     -> keep only the first one (lowest id)
# MANUAL_DROP is kept for single ids to remove after visual review.
# ---------------------------------------------------------------
MANUAL_DROP = []   # e.g. ["train_00713", "train_00854"]

clean["drop_reason"] = ""
n_labels = clean.groupby("group_id")["label"].transform("nunique")
group_size = clean.groupby("dup_id")["id"].transform("size")

conflict = n_labels > 1
clean.loc[conflict, "drop_reason"] = "conflicting copies (>= 0.97)"

extra_copy = (group_size > 1) & ~conflict & clean.duplicated("dup_id", keep="first")
clean.loc[extra_copy, "drop_reason"] = "extra copy (>= 0.99)"

manual = clean["id"].isin(MANUAL_DROP) & (clean["drop_reason"] == "")
clean.loc[manual, "drop_reason"] = "manual review"

print("Removed images by reason:")
display(clean.loc[clean["drop_reason"] != "", "drop_reason"].value_counts().rename("images").to_frame())

Removed images by reason:


,images
drop_reason,
extra copy (>= 0.99),174
conflicting copies (>= 0.97),63


In [9]:
# ---------------------------------------------------------------
# 5.3 'noisy' flag: images with strong chromatic noise (colorfulness
# robust z-score > COLOR_Z). They are KEPT, but the flag is stored to
# evaluate validation F1 separately on noisy / clean images.
# ---------------------------------------------------------------
clean["noisy"] = valid.loc[clean.index, "colorfulness_z"].to_numpy() > COLOR_Z

noisy_table = pd.crosstab(clean["label"].map(CLASS_NAMES), clean["noisy"])
noisy_table = noisy_table.reindex(list(CLASS_NAMES.values()), fill_value=0)
noisy_table["noisy_%"] = 100 * noisy_table.get(True, 0) / noisy_table.sum(axis=1)
display(noisy_table.round(2))
print(f"Noisy training images: {int(clean['noisy'].sum())} of {len(clean)}")

noisy,False,True,noisy_%
label,,,
Spaghetti alla rinfusa,213,0,0.00
Gemelli,345,0,0.00
Gnocchi,651,93,12.50
Ravioli,570,115,16.79
Penne,83,0,0.00
Fusilli con la sbarra,510,74,12.67
Fusilli stretti,497,68,12.04
Riccioli,477,0,0.00
Linguine,275,1,0.36


Noisy training images: 351 of 4328


In [10]:
# ---------------------------------------------------------------
# 5.4 Final cleaned training table: class counts before/after.
# ---------------------------------------------------------------
train_clean = clean[clean["drop_reason"] == ""][["id", "label", "group_id", "noisy"]].reset_index(drop=True)

before = clean["label"].value_counts().reindex(list(CLASS_NAMES), fill_value=0)
after = train_clean["label"].value_counts().reindex(list(CLASS_NAMES), fill_value=0)
class_change = pd.DataFrame({"class": list(CLASS_NAMES.values()),
                             "before": before.to_numpy(), "after": after.to_numpy()})
class_change["removed"] = class_change["before"] - class_change["after"]
display(class_change)
print(f"Training images: {len(clean)} -> {len(train_clean)} "
      f"({len(clean) - len(train_clean)} removed), {train_clean['group_id'].nunique()} groups")

,class,before,after,removed
0,Spaghetti alla rinfusa,213,212,1
1,Gemelli,345,344,1
2,Gnocchi,744,671,73
3,Ravioli,685,617,68
4,Penne,83,82,1
5,Fusilli con la sbarra,584,544,40
6,Fusilli stretti,565,517,48
7,Riccioli,477,475,2
8,Linguine,276,275,1
9,Lasagna,356,354,2


Training images: 4328 -> 4091 (237 removed), 4027 groups


## 6. Cross-validation folds
No validation set is given, so the cleaned training set is split into **5 folds**: each model is trained on 4 folds and evaluated (weighted F1) on the remaining one.

`StratifiedGroupKFold` combines two constraints:
- **Stratified**: every fold keeps about the same class proportions, so even the rarest class (Penne) appears in each validation fold.
- **Group**: all images with the same `group_id` (copies and near-copies) stay in the same fold. A copy in training and its twin in validation would make the validation score too optimistic.

With 5 folds each validation set has ~820 images (a stable F1 estimate) and each training run uses 80% of the data. The result is saved as `train_folds.csv`, the starting point for training.

In [11]:
# ---------------------------------------------------------------
# Assign each cleaned training image to one of N_FOLDS folds with
# StratifiedGroupKFold (labels balanced, copies kept together).
# ---------------------------------------------------------------
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

train_clean["fold"] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train_clean, y=train_clean["label"], groups=train_clean["group_id"])):
    train_clean.loc[val_idx, "fold"] = fold   # val_idx are positions; the index is 0..n-1 after reset_index

assert (train_clean["fold"] >= 0).all(), "some images were not assigned to a fold"
assert (train_clean.groupby("group_id")["fold"].nunique() == 1).all(), "a group is split across folds"

# Images per class in each validation fold (should be roughly constant along each row)
fold_table = pd.crosstab(train_clean["label"].map(CLASS_NAMES), train_clean["fold"])
fold_table = fold_table.reindex(list(CLASS_NAMES.values()))
fold_table.loc["TOTAL"] = fold_table.sum()
display(fold_table)

FOLDS_CSV = DATA_DIR / "train_folds.csv"
train_clean.to_csv(FOLDS_CSV, index=False)
print(f"Saved {FOLDS_CSV}")

fold,0,1,2,3,4
label,,,,,
Spaghetti alla rinfusa,57,44,33,39,39
Gemelli,71,64,60,68,81
Gnocchi,143,121,138,133,136
Ravioli,109,127,131,123,127
Penne,22,9,12,18,21
Fusilli con la sbarra,108,96,114,111,115
Fusilli stretti,103,117,91,116,90
Riccioli,90,92,106,100,87
Linguine,53,63,64,45,50


Saved /content/train_folds.csv
